# 02 — Dữ liệu vĩ mô, ngành và tài chính

Tương ứng src/fetch_supplemental.py. Tách việc xem nguồn, chuẩn hóa, căn ngày có thể biết và kiểm tra dữ liệu thiếu.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Các CSV đã đóng gói trong data/raw/supplemental/ và dữ liệu giá FPT.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Xem các tệp nguồn

Nguồn gồm CSV xuất của nhà cung cấp và các mốc cổ phiếu lưu hành. Đây là quy trình tái chạy snapshot offline; ô này không khẳng định đã tải mới các website.

In [ ]:
source_files = sorted(RAW_SUPPLEMENTAL_DIR.glob("*.csv"))
display(pd.DataFrame([{"Tệp": p.name, "Số byte": p.stat().st_size} for p in source_files]))
source_info = json.loads((ROOT / "data/sources.json").read_text(encoding="utf-8"))
display(source_info)

## 5. Chạy chuẩn hóa và ghép lùi

Chuyển định dạng ngày, dấu phân cách số, đơn vị vốn hóa; tính EBIT bằng lợi nhuận trước thuế + chi phí lãi vay. Dữ liệu chỉ xuất hiện sau ngày hiệu lực, có giới hạn tuổi dữ liệu; không backfill.

In [ ]:
from fetch_supplemental import run_from_cache, MODEL_SUPPLEMENTAL_COLUMNS
# Chuẩn hóa CSV đã đóng gói; không tải mạng và không nội suy từ tương lai.
supplement_audit = run_from_cache(RAW_SUPPLEMENTAL_DIR, SUPPLEMENTAL_PATH, end=CUTOFF)
supplement = pd.read_csv(SUPPLEMENTAL_PATH, parse_dates=["date"])
display(pd.DataFrame(supplement_audit["source_audit"]).T)
print("Đã ghi:", SUPPLEMENTAL_PATH.relative_to(ROOT))

## 6. Xem độ trễ và khoảng trống

Báo cáo quý +45 ngày, GDP năm +180 ngày là giả định; dữ liệu ngày/CPI/ngành/cổ phiếu +1 ngày. Trống trước ngày có thể biết là điều bình thường; không dùng báo cáo sau để lấp ngược.

In [ ]:
display(pd.Series(supplement_audit["release_delays_calendar_days"]).to_frame("Số ngày trễ"))
display(supplement[MODEL_SUPPLEMENTAL_COLUMNS].isna().sum().sort_values(ascending=False).to_frame("Phiên thiếu"))
display(supplement[["date", *MODEL_SUPPLEMENTAL_COLUMNS]].tail())

## 7. Ví dụ nhỏ: thông tin chưa công bố

Một giá trị được ghi cho ngày 02/01 nhưng chỉ cho phép dùng sau 1 ngày. Ngày 02/01 phải còn thiếu; từ 03/01 mới được dùng. Đây là phép ghép lùi theo thời điểm có thể biết.

In [ ]:
from fetch_supplemental import release_aware_asof
sessions = pd.Series(pd.to_datetime(["2025-01-02", "2025-01-03", "2025-01-06"]))
tiny_source = pd.DataFrame({"time": pd.to_datetime(["2025-01-02"]), "example_value": [100.0]})
tiny_aligned = release_aware_asof(sessions, tiny_source, delay_days=1, value_columns=["example_value"], max_age_days=10)
display(tiny_aligned)
assert pd.isna(tiny_aligned["example_value"].iloc[0])
assert tiny_aligned["example_value"].iloc[1] == 100.0

**Đầu ra:** data/processed/fpt_multisource_features.csv và reports/supplemental_data_audit.json. Tiếp theo: 03_eda.ipynb.